# Chapter 6: Precognition (Thinking Step by Step)

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `get_completion` helper function.

In [2]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 38.1 MB/s eta 0:00:00


---

## Lesson

If someone woke you up and immediately started asking you several complicated questions that you had to respond to right away, how would you do? Probably not as good as if you were given time to **think through your answer first**.

Guess what? Claude is the same way.

**Giving Claude time to think step by step sometimes makes Claude more accurate**, particularly for complex tasks. However, **thinking only counts when it's out loud**. You cannot ask Claude to think but output only the answer - in this case, no thinking has actually occurred.

### Examples

In the prompt below, it's clear to a human reader that the second sentence belies the first. But **Claude takes the word "unrelated" too literally**.

In [3]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

The sentiment is **negative/sarcastic**.

While the first sentence appears positive on the surface ("blew my mind," "freshness and originality"), the second sentence reveals this is sarcasm. By claiming to have lived under a rock since 1900, the reviewer is implying that the movie is actually *not* fresh or original at all—it's so derivative that only someone completely out of touch with modern cinema would find it novel.

The review is using irony to criticize the movie.


To improve Claude's response, let's **allow Claude to think things out first before answering**. We do that by literally spelling out the steps that Claude should take in order to process and think through its task. Along with a dash of role prompting, this empowers Claude to understand the review more deeply.

In [4]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

<positive-argument>
The reviewer explicitly states the movie "blew my mind" and praises it for "freshness and originality" - these are strong positive descriptors that, taken at face value, indicate genuine enthusiasm for the film.
</positive-argument>

<negative-argument>
The phrase "In totally unrelated news, I have been living under a rock since 1900" is sarcasm that undermines the positive statement. The reviewer is implying they have such poor taste or limited exposure to cinema that their praise is unreliable. This self-deprecating joke suggests the movie is likely derivative or unoriginal, contradicting their stated opinion. The "totally unrelated" framing makes it clear this is directly related to—and meant to mock—their previous claim.
</negative-argument>

**Answer: NEGATIVE**

This is a sarcastic review. The reviewer is using irony to criticize the movie by praising it while simultaneously suggesting their judgment is worthless due to extreme isolation from culture. The impl

**Claude is sometimes sensitive to ordering**. This example is on the frontier of Claude's ability to understand nuanced text, and when we swap the order of the arguments from the previous example so that negative is first and positive is second, this changes Claude's overall assessment to positive.

In most situations (but not all, confusingly enough), **Claude is more likely to choose the second of two options**, possibly because in its training data from the web, second options were more likely to be correct.

In [5]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

<negative-argument>
The reviewer admits to living under a rock since 1900, which undermines their credibility as a judge of what is actually "fresh" or "original." This suggests they lack the context and exposure to other films needed to make a meaningful comparison. The phrase "unrelatedly" is clearly sarcastic, implying the opposite—that it IS related and explains why they found the movie so novel. This self-aware admission suggests the positive praise may not be trustworthy.
</negative-argument>

<positive-argument>
The reviewer uses explicitly positive language: "blew my mind," "freshness," and "originality." These are strong compliments about the movie itself. Even if their credibility is questionable due to limited exposure, the literal sentiment expressed about the film is enthusiastically positive.
</positive-argument>

**Answer: The sentiment is POSITIVE (in literal terms) but SATIRICAL/SARCASTIC in intent.**

The review is written as positive on the surface, but the sarcastic

**Letting Claude think can shift Claude's answer from incorrect to correct**. It's that simple in many cases where Claude makes mistakes!

Let's go through an example where Claude's answer is incorrect to see how asking Claude to think can fix that.

In [6]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

# Gladiator (2000)

This film stars Russell Crowe, who was born in 1964. Let me correct that.

**Forrest Gump (1994)** stars Tom Hanks, who was born in 1956.


Let's fix this by asking Claude to think step by step, this time in `<brainstorm>` tags.

In [7]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))

<brainstorm>
Let me think about famous actors born in 1956:
- Tom Cruise - born December 3, 1962 (not 1956)
- Eddie Murphy - born April 3, 1961 (not 1956)
- Patrick Swayze - born August 18, 1952 (not 1956)
- Tom Hanks - born July 9, 1956 ✓
- Mel Gibson - born January 3, 1956 ✓
- Kurt Russell - born March 17, 1951 (not 1956)

Great! Tom Hanks and Mel Gibson were both born in 1956.

Tom Hanks famous movies: Forrest Gump, Cast Away, Saving Private Ryan, Toy Story
Mel Gibson famous movies: Braveheart, Mad Max, Lethal Weapon
</brainstorm>

**Forrest Gump** (1994) - starring Tom Hanks, who was born in 1956.


If you would like to experiment with the lesson prompts without changing any content above, scroll all the way to the bottom of the lesson notebook to visit the [**Example Playground**](#example-playground).

---

## Exercises
- [Exercise 6.1 - Classifying Emails](#exercise-61---classifying-emails)
- [Exercise 6.2 - Email Classification Formatting](#exercise-62---email-classification-formatting)

### Exercise 6.1 - Classifying Emails
In this exercise, we'll be instructing Claude to sort emails into the following categories:
- (A) Pre-sale question
- (B) Broken or defective item
- (C) Billing question
- (D) Other (please explain)

For the first part of the exercise, change the `PROMPT` to **make Claude output the correct classification and ONLY the classification**. Your answer needs to **include the letter (A - D) of the correct choice, with the parentheses, as well as the name of the category**.

Refer to the comments beside each email in the `EMAILS` list to know which category that email should be classified under.

In [11]:
# Prompt template with a placeholder for the variable content
PROMPT = """
Please classify this emails according to the categories: {email}
<categories>
(A) Pre-sale question
(B) Broken or defective item
(C) Billing question
(D) Other (please explain)
</categories>
"""

# Prefill for Claude's response, if any
PREFILL = "the answer should include only the classification"

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "A\) P",
    "B": "B\) B",
    "C": "C\) B",
    "D": "D\) O"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):

    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)

    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])

    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

<>:33: SyntaxWarning: invalid escape sequence '\)'
<>:34: SyntaxWarning: invalid escape sequence '\)'
<>:35: SyntaxWarning: invalid escape sequence '\)'
<>:36: SyntaxWarning: invalid escape sequence '\)'
<>:33: SyntaxWarning: invalid escape sequence '\)'
<>:34: SyntaxWarning: invalid escape sequence '\)'
<>:35: SyntaxWarning: invalid escape sequence '\)'
<>:36: SyntaxWarning: invalid escape sequence '\)'
/tmp/ipykernel_356/2500373515.py:33: SyntaxWarning: invalid escape sequence '\)'
  "A": "A\) P",
/tmp/ipykernel_356/2500373515.py:34: SyntaxWarning: invalid escape sequence '\)'
  "B": "B\) B",
/tmp/ipykernel_356/2500373515.py:35: SyntaxWarning: invalid escape sequence '\)'
  "C": "C\) B",
/tmp/ipykernel_356/2500373515.py:36: SyntaxWarning: invalid escape sequence '\)'
  "D": "D\) O"


--------------------------- Full prompt with variable substutions ---------------------------
USER TURN

Please classify this emails according to the categories: Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.
<categories>
(A) Pre-sale question
(B) Broken or defective item
(C) Billing question
(D) Other (please explain)
</categories>


ASSISTANT TURN
the answer should include only the classification

------------------------------------- Claude's response -------------------------------------
 letter and category name.

**Classification: (B) Broken or defective item**

The email describes a product malfunction (strange noise, burning smell) and requests a replacement, which are clear indicators of a defective item.

------------------------------------------ GRADING ------------------------------------------
This exercise has been correctly solved: True 






-------------

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_6_1_hint; print(exercise_6_1_hint)

Still stuck? Run the cell below for an example solution.

In [ ]:
from hints import exercise_6_1_solution; print(exercise_6_1_solution)

### Exercise 6.2 - Email Classification Formatting
In this exercise, we're going to refine the output of the above prompt to yield an answer formatted exactly how we want it.

Use your favorite output formatting technique to make Claude wrap JUST the letter of the correct classification in `<answer></answer>` tags. For instance, the answer to the first email should contain the exact string `<answer>B</answer>`.

Refer to the comments beside each email in the `EMAILS` list if you forget which letter category is correct for each email.

In [13]:
# Prompt template with a placeholder for the variable content
PROMPT = """Please classify this email according to the categories: {email}
<categories>
(A) Pre-sale question
(B) Broken or defective item
(C) Billing question
(D) Other (please explain)
</categories>"""

# Prefill for Claude's response, if any
PREFILL = "answer only the category's letter code and put it in tags <answer></answer>"

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "<answer>A</answer>",
    "B": "<answer>B</answer>",
    "C": "<answer>C</answer>",
    "D": "<answer>D</answer>"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):

    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)

    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])

    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

--------------------------- Full prompt with variable substutions ---------------------------
USER TURN
Please classify this email according to the categories: Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.
<categories>
(A) Pre-sale question
(B) Broken or defective item
(C) Billing question
(D) Other (please explain)
</categories>

ASSISTANT TURN
answer only the category's letter code and put it in tags <answer></answer>

------------------------------------- Claude's response -------------------------------------


<answer>B</answer>

**Reasoning:** The customer is reporting that their Mixmaster4000 is producing strange noises, has a burning smell, and is requesting a replacement. These are clear indicators of a defective/broken product, which falls under category (B).

------------------------------------------ GRADING ------------------------------------------
This exer

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_6_2_hint; print(exercise_6_2_hint)

### Congrats!

If you've solved all exercises up until this point, you're ready to move to the next chapter. Happy prompting!

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson and tweak prompts to see how it may affect Claude's responses.

In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))